# 02 · SVD and low-rank approximation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/math/02_svd_low_rank.ipynb) &nbsp; 📖 Concept deck: [Eigenvalues, SVD and low rank](https://nilesh-salpe.github.io/visual-ai/math/03-eigen-svd-low-rank/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** the truncated SVD, the energy captured by the top singular values, the Eckart–Young error of the best rank-r approximation, and the parameter count of a LoRA update.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


rng = np.random.default_rng(0)
def low_rank_plus_noise(m, n, r, noise=0.01, seed=0):
    g = np.random.default_rng(seed)
    return g.normal(size=(m, r)) @ g.normal(size=(r, n)) + noise * g.normal(size=(m, n))

## Task 1: Best rank-r approximation

Use `np.linalg.svd(W, full_matrices=False)` to get $W = U \Sigma V^\top$, keep the top `r` singular values and vectors, and return the rank-r matrix $U_r \Sigma_r V_r^\top$.
By the Eckart–Young theorem this is the closest rank-r matrix to W in Frobenius norm.

In [ ]:
def rank_r(W, r):
    # TODO: truncated SVD reconstruction
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_rank_r(f):
    W = low_rank_plus_noise(40, 30, 5)
    A = f(W, 5)
    assert A.shape == W.shape
    assert np.linalg.matrix_rank(A, tol=1e-8) == 5, "result must have rank r"
    s = np.linalg.svd(W, compute_uv=False)
    err = np.linalg.norm(W - A)
    assert np.isclose(err, np.sqrt(np.sum(s[5:] ** 2))), "Eckart–Young: error² = sum of the discarded σ²"
    assert np.allclose(f(W, 30), W), "full rank must reproduce W"

check("rank_r", _test_rank_r, rank_r)

## Task 2: Energy captured by the top r singular values

Return the fraction $\sum_{i \le r} \sigma_i^2 / \sum_i \sigma_i^2$ for every r = 1 … min(m, n) as an array.
For matrices with low-rank structure the curve jumps to almost 1 after a few components.

In [ ]:
def energy(W):
    # TODO: cumulative sum of squared singular values, normalised
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_energy(f):
    e = f(low_rank_plus_noise(50, 40, 3))
    assert len(e) == 40 and np.isclose(e[-1], 1)
    assert np.all(np.diff(e) >= -1e-12), "energy must be non-decreasing"
    assert e[2] > 0.99, f"a rank-3 + noise matrix should have >99% energy in 3 components, got {e[2]:.3f}"
    er = f(np.random.default_rng(3).normal(size=(50, 40)))
    assert er[2] < 0.3, "a random matrix spreads energy over many components"

check("energy", _test_energy, energy)

## Task 3: Parameters in a LoRA update

LoRA replaces a full update $\Delta W \in \mathbb{R}^{d \times k}$ with $BA$, where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$.
Return `(full, lora, ratio)`: the parameter counts of $\Delta W$ and of $(B, A)$, and lora / full.

In [ ]:
def lora_params(d, k, r):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_lora_params(f):
    full, lora, ratio = f(4096, 4096, 8)
    assert full == 4096 * 4096 and lora == 8 * (4096 + 4096)
    assert np.isclose(ratio, 65536 / 16777216)
    assert f(10, 20, 1)[1] == 30

check("lora_params", _test_lora_params, lora_params)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Best rank-r approximation
def rank_r_solution(W, r):
    U, s, Vt = np.linalg.svd(W, full_matrices=False)
    return (U[:, :r] * s[:r]) @ Vt[:r]

check("rank_r (reference)", _test_rank_r, rank_r_solution)

In [ ]:
# @title Solution for Task 2: Energy captured by the top r singular values
def energy_solution(W):
    s2 = np.linalg.svd(W, compute_uv=False) ** 2
    return np.cumsum(s2) / s2.sum()

check("energy (reference)", _test_energy, energy_solution)

In [ ]:
# @title Solution for Task 3: Parameters in a LoRA update
def lora_params_solution(d, k, r):
    full, lora = d * k, r * (d + k)
    return full, lora, lora / full

check("lora_params (reference)", _test_lora_params, lora_params_solution)

---
## Experiments
**Compressing a matrix.** Storing the rank-r factors of an m×n matrix costs r(m+n) numbers instead of mn. Watch the error fall as r grows for a matrix with real low-rank structure.

In [ ]:
W = low_rank_plus_noise(256, 256, 16, noise=0.05, seed=2)
for r in [1, 4, 8, 16, 32, 64]:
    A = rank_r_solution(W, r)
    print(f"r={r:3d}: relative error {np.linalg.norm(W - A) / np.linalg.norm(W):.4f}   storage {r * 512 / 65536:6.1%} of the full matrix")

---
## Takeaways
* The SVD writes any matrix as a sum of rank-1 pieces ordered by importance (singular values).
* Truncating it gives the best rank-r approximation, with squared error equal to the discarded σ².
* LoRA bets that fine-tuning updates are low rank: r(d+k) parameters instead of d·k.